# OD Aggregation & Graph Construction

In [16]:
import pandas as pd
import numpy as np
import networkx as nx

df = pd.read_csv("../data/delivery_data.csv")

print(df.shape)

(144867, 24)


## Datetime Processing

In [17]:
date_cols = [
    "trip_creation_time",
    "od_start_time",
    "od_end_time",
    "cutoff_timestamp"
]

for col in date_cols:
    df[col] = pd.to_datetime(
        df[col],
        format="mixed"
    )

df[date_cols].head()

,trip_creation_time,od_start_time,od_end_time,cutoff_timestamp
0,2018-09-20 02:35:36.476840,2018-09-20 03:21:32.418600,2018-09-20 04:47:45.236797,2018-09-20 04:27:55.000000
1,2018-09-20 02:35:36.476840,2018-09-20 03:21:32.418600,2018-09-20 04:47:45.236797,2018-09-20 04:17:55.000000
2,2018-09-20 02:35:36.476840,2018-09-20 03:21:32.418600,2018-09-20 04:47:45.236797,2018-09-20 04:01:19.505586
3,2018-09-20 02:35:36.476840,2018-09-20 03:21:32.418600,2018-09-20 04:47:45.236797,2018-09-20 03:39:57.000000
4,2018-09-20 02:35:36.476840,2018-09-20 03:21:32.418600,2018-09-20 04:47:45.236797,2018-09-20 03:33:55.000000


In [18]:
print(df[date_cols].dtypes)

trip_creation_time    datetime64[us]
od_start_time         datetime64[us]
od_end_time           datetime64[us]
cutoff_timestamp      datetime64[us]
dtype: object


## OD-Level Aggregation

In [19]:
agg_dict = {
    "route_type": "first",
    "trip_creation_time": "first",
    "od_start_time": "first",
    "od_end_time": "first",
    "actual_time": "max",
    "osrm_time": "max",
    "osrm_distance": "max",
    "actual_distance_to_destination": "max",
    "data": "first"
}

leg_df = (
    df.groupby(
        [
            "trip_uuid",
            "source_center",
            "destination_center"
        ],
        as_index=False
    )
    .agg(agg_dict)
)

print("Leg Shape:", leg_df.shape)

leg_df.head()

Leg Shape: (26368, 12)


,trip_uuid,source_center,destination_center,route_type,trip_creation_time,od_start_time,od_end_time,actual_time,osrm_time,osrm_distance,actual_distance_to_destination,data
0,trip-153671041653548748,IND209304AAA,IND000000ACB,FTL,2018-09-12 00:00:16.535741,2018-09-12 16:39:46.858469,2018-09-13 13:40:23.123744,732.0,349.0,446.5496,383.759164,training
1,trip-153671041653548748,IND462022AAA,IND209304AAA,FTL,2018-09-12 00:00:16.535741,2018-09-12 00:00:16.535741,2018-09-12 16:39:46.858469,830.0,394.0,544.8027,440.973689,training
2,trip-153671042288605164,IND561203AAB,IND562101AAA,Carting,2018-09-12 00:00:22.886430,2018-09-12 02:03:09.655591,2018-09-12 03:01:59.598855,47.0,26.0,28.1994,24.644021,training
3,trip-153671042288605164,IND572101AAA,IND561203AAB,Carting,2018-09-12 00:00:22.886430,2018-09-12 00:00:22.886430,2018-09-12 02:03:09.655591,96.0,42.0,56.9116,48.542890,training
4,trip-153671043369099517,IND000000ACB,IND160002AAC,FTL,2018-09-12 00:00:33.691250,2018-09-14 03:40:17.106733,2018-09-14 17:34:55.442454,611.0,212.0,281.2109,242.309306,training


In [20]:
leg_df["delay_ratio"] = (
    leg_df["actual_time"]
    /
    leg_df["osrm_time"]
)

leg_df["sla_breach"] = (
    leg_df["actual_time"]
    >
    1.2 * leg_df["osrm_time"]
)

leg_df.head()

,trip_uuid,source_center,destination_center,route_type,trip_creation_time,od_start_time,od_end_time,actual_time,osrm_time,osrm_distance,actual_distance_to_destination,data,delay_ratio,sla_breach
0,trip-153671041653548748,IND209304AAA,IND000000ACB,FTL,2018-09-12 00:00:16.535741,2018-09-12 16:39:46.858469,2018-09-13 13:40:23.123744,732.0,349.0,446.5496,383.759164,training,2.097421,True
1,trip-153671041653548748,IND462022AAA,IND209304AAA,FTL,2018-09-12 00:00:16.535741,2018-09-12 00:00:16.535741,2018-09-12 16:39:46.858469,830.0,394.0,544.8027,440.973689,training,2.106599,True
2,trip-153671042288605164,IND561203AAB,IND562101AAA,Carting,2018-09-12 00:00:22.886430,2018-09-12 02:03:09.655591,2018-09-12 03:01:59.598855,47.0,26.0,28.1994,24.644021,training,1.807692,True
3,trip-153671042288605164,IND572101AAA,IND561203AAB,Carting,2018-09-12 00:00:22.886430,2018-09-12 00:00:22.886430,2018-09-12 02:03:09.655591,96.0,42.0,56.9116,48.542890,training,2.285714,True
4,trip-153671043369099517,IND000000ACB,IND160002AAC,FTL,2018-09-12 00:00:33.691250,2018-09-14 03:40:17.106733,2018-09-14 17:34:55.442454,611.0,212.0,281.2109,242.309306,training,2.882075,True


In [21]:
print(
    leg_df["delay_ratio"]
    .describe()
)

count    26368.000000
mean         2.540968
std          2.606550
min          0.387755
25%          1.647059
50%          2.000000
75%          2.593750
max         77.387097
Name: delay_ratio, dtype: float64


In [22]:
sla_rate = (
    leg_df["sla_breach"]
    .mean()
    * 100
)

print(
    f"OD-Level SLA Breach Rate: {sla_rate:.2f}%"
)

OD-Level SLA Breach Rate: 94.66%


In [23]:
print("Raw Rows:", len(df))

print("Unique Trips:",
      df["trip_uuid"].nunique())

print("OD Legs:",
      leg_df.shape[0])

print(
    "Avg Rows per OD Leg:",
    len(df) / leg_df.shape[0]
)

Raw Rows: 144867
Unique Trips: 14817
OD Legs: 26368
Avg Rows per OD Leg: 5.494045813106796


In [24]:
leg_df[[
    "actual_time",
    "osrm_time",
    "delay_ratio"
]].describe()

,actual_time,osrm_time,delay_ratio
count,26368.000000,26368.000000,26368.000000
mean,200.690193,91.072853,2.540968
std,384.853640,185.790830,2.606550
min,9.000000,6.000000,0.387755
25%,51.000000,25.000000,1.647059
50%,84.000000,39.000000,2.000000
75%,168.000000,73.000000,2.593750
max,4532.000000,1686.000000,77.387097


In [25]:
leg_df.sort_values(
    "delay_ratio",
    ascending=False
)[[
    "trip_uuid",
    "source_center",
    "destination_center",
    "actual_time",
    "osrm_time",
    "delay_ratio"
]].head(10)

,trip_uuid,source_center,destination_center,actual_time,osrm_time,delay_ratio
13399,trip-153761568647456368,IND271001AAA,IND271201AAA,2399.0,31.0,77.387097
5359,trip-153706705984334815,IND211008AAA,IND211002AAB,1082.0,15.0,72.133333
3134,trip-153691774924973666,IND395006AAA,IND395023AAD,770.0,11.0,70.000000
15545,trip-153778216596113868,IND452001AAC,IND425201AAA,1195.0,18.0,66.388889
6751,trip-153717640309053191,IND452001AAC,IND425201AAA,1138.0,18.0,63.222222
1742,trip-153682944027974510,IND211008AAA,IND211002AAB,1006.0,16.0,62.875000
21551,trip-153820050595464971,IND416606AAA,IND416510AAA,2664.0,46.0,57.913043
20188,trip-153809667984245845,IND842003AAB,IND482002AAA,441.0,8.0,55.125000
19000,trip-153800794553157586,IND208012AAA,IND209304AAA,878.0,17.0,51.647059
4486,trip-153700839622813110,IND271001AAA,IND271210AAA,2536.0,50.0,50.720000


In [26]:
train_df = leg_df[
    leg_df["data"] == "training"
].copy()

print("Training Legs:", train_df.shape)

Training Legs: (18947, 14)


In [27]:
import networkx as nx

G = nx.DiGraph()

for (src, dst), group in train_df.groupby(
    ["source_center", "destination_center"]
):

    delay_ratio = (
        group["actual_time"].median()
        /
        group["osrm_time"].median()
    )

    G.add_edge(
        src,
        dst,
        weight=delay_ratio,
        trip_count=len(group)
    )

print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

Nodes: 1590
Edges: 2508


In [28]:
density = nx.density(G)

print("Density:", density)

Density: 0.0009926736882102188


In [29]:
largest_component = max(
    nx.weakly_connected_components(G),
    key=len
)

print(
    "Largest Component Size:",
    len(largest_component)
)

Largest Component Size: 1263
